# VisionTrack — Data Exploration & Model Analysis

This notebook explores the COCO dataset structure and analyzes YOLO26 model characteristics.

**Sections:**
1. COCO Dataset Overview
2. Class Distribution Analysis
3. Model Architecture & Parameters
4. ONNX Model Inspection
5. Sample Inference Visualization

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# COCO class names
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train",
    "truck", "boat", "traffic light", "fire hydrant", "stop sign",
    "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep",
    "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
    "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard",
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard",
    "surfboard", "tennis racket", "bottle", "wine glass", "cup", "fork",
    "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv",
    "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave",
    "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase",
    "scissors", "teddy bear", "hair drier", "toothbrush",
]

print(f'COCO classes: {len(COCO_CLASSES)}')
print(f'First 10: {COCO_CLASSES[:10]}')

## 1. COCO Dataset Overview

COCO (Common Objects in Context) is the standard benchmark for object detection.
- 80 object categories
- 330K images with 1.5M annotations
- Complex scenes with multiple overlapping objects

In [ ]:
# COCO class frequency distribution (approximate, from official stats)
# These are the relative frequencies of each class in the training set
coco_freq = {
    'person': 26.0, 'car': 7.5, 'truck': 3.5, 'backpack': 1.5, 'umbrella': 1.5,
    'handbag': 1.5, 'tie': 1.0, 'suitcase': 0.8, 'frisbee': 0.5, 'skis': 0.5,
    'sports ball': 0.5, 'kite': 0.5, 'baseball bat': 0.3, 'baseball glove': 0.3,
    'skateboard': 0.3, 'surfboard': 0.3, 'tennis racket': 0.3, 'bottle': 1.0,
    'wine glass': 0.8, 'cup': 1.0, 'fork': 0.3, 'knife': 0.3, 'spoon': 0.3,
    'bowl': 0.8, 'banana': 0.3, 'apple': 0.3, 'sandwich': 0.3, 'orange': 0.3,
    'broccoli': 0.3, 'carrot': 0.3, 'hot dog': 0.3, 'pizza': 0.5, 'donut': 0.3,
    'cake': 0.5, 'chair': 2.5, 'couch': 1.0, 'potted plant': 0.8, 'bed': 0.5,
    'dining table': 1.5, 'toilet': 0.5, 'tv': 1.0, 'laptop': 0.8, 'mouse': 0.3,
    'remote': 0.3, 'keyboard': 0.3, 'cell phone': 0.8, 'microwave': 0.3,
    'oven': 0.3, 'toaster': 0.1, 'sink': 0.3, 'refrigerator': 0.3,
    'book': 0.5, 'clock': 0.5, 'vase': 0.5, 'scissors': 0.3,
    'teddy bear': 0.3, 'hair drier': 0.1, 'toothbrush': 0.1,
}

# Top 20 most common classes
sorted_freq = sorted(coco_freq.items(), key=lambda x: x[1], reverse=True)[:20]
names, freqs = zip(*sorted_freq)

plt.figure(figsize=(12, 6))
plt.barh(range(len(names)), freqs, color='steelblue')
plt.yticks(range(len(names)), names)
plt.xlabel('Relative Frequency (%)')
plt.title('COCO Dataset — Top 20 Most Common Classes')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 2. YOLO26 Model Analysis

YOLO26 variants (Ultralytics, 2026):

| Model | Params | mAP (val) | Latency (CPU) |
|-------|--------|-----------|----------------|
| YOLO26n | 2.4M | 40.9 | ~18ms |
| YOLO26s | 7.2M | 47.2 | ~25ms |
| YOLO26m | 20.1M | 51.5 | ~45ms |
| YOLO26l | 26.3M | 53.1 | ~60ms |
| YOLO26x | 56.9M | 55.2 | ~100ms |

We use **YOLO26n** (nano) for real-time inference.

In [ ]:
# Inspect the ONNX model
import onnxruntime as ort
from pathlib import Path

model_path = Path('../models/yolo26n.onnx')
if model_path.exists():
    session = ort.InferenceSession(str(model_path))
    
    inp = session.get_inputs()[0]
    out = session.get_outputs()[0]
    
    print(f'Model: {model_path.name}')
    print(f'  Input:  {inp.name}  shape={inp.shape}  dtype={inp.type}')
    print(f'  Output: {out.name}  shape={out.shape}  dtype={out.type}')
    print(f'  Size:   {model_path.stat().st_size / (1024*1024):.1f} MB')
    print(f'  Providers: {session.get_providers()}')
else:
    print(f'Model not found at {model_path}')
    print('Run: python python/export_onnx.py --weights yolo26n.pt')

## 3. Sample Inference

Run detection on a sample image and visualize the results.

In [ ]:
from detector import Detector
from utils.preprocessing import Preprocessor
from utils.visualization import draw_detections

# Load model
detector = Detector('../models/yolo26n.onnx', num_classes=80)
preprocessor = Preprocessor(detector.input_size)

# Load test image
test_images = list(Path('../assets').glob('*.jpg')) + list(Path('../data').glob('*.jpg'))
if test_images:
    img_path = test_images[0]
    image = cv2.imread(str(img_path))
    print(f'Image: {img_path.name}  shape: {image.shape}')
    
    # Detect
    tensor, scale, padding = preprocessor.preprocess(image)
    output = detector.infer(tensor)
    boxes_model, scores, class_ids = detector.postprocess(output)
    boxes = preprocessor.scale_boxes(boxes_model, scale, padding, image.shape)
    
    # Visualize
    annotated = draw_detections(image, boxes, scores, class_ids)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'Detected {len(boxes)} objects')
    plt.show()
else:
    print('No test images found. Place a .jpg in assets/ or data/')

## 4. Bounding Box Visualization

Visualize raw detections with confidence scores and class labels.

In [ ]:
# Print detection details
print(f'\nDetections ({len(boxes)} total):')
print(f'{"Class":<15} {"Confidence":<12} {"Box (x1,y1,x2,y2)":<25}')
print('-' * 55)

for i in range(len(boxes)):
    name = COCO_CLASSES[int(class_ids[i])] if int(class_ids[i]) < len(COCO_CLASSES) else f'cls_{int(class_ids[i])}'
    box = boxes[i]
    print(f'{name:<15} {scores[i]:<12.3f} ({box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f})')